In [38]:
from typing import TypeAlias,List
import multiprocessing as mp
import regex as re
from typing import BinaryIO
import time
from collections import defaultdict
import os

Vocab: TypeAlias = dict[int,bytes]  # Vocabulary (int -> bytes)
Merges: TypeAlias = list[tuple[bytes,bytes]] 
BPEResult: TypeAlias = tuple[Vocab, Merges]

In [27]:
## 环境配置
FILE_PATH = "../data/TinyStoriesV2-GPT4-train.txt"
VOCAB_SIZE = 300
NUM_PROCESSES = 32
SPECIAL_TOKENS = b"<|endoftext|>"

SPECIAL_TOKEN_STR = SPECIAL_TOKENS.decode("utf-8", errors="ignore")
SPECIAL_TOKEN_PATTERN = re.compile(re.escape(SPECIAL_TOKEN_STR))

# ===== 2. 通用 token 正则：GPT-2 风格的简化版 =====
GENERAL_TOKEN_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d|   # 常见英文缩写
        \p{L}+|                   # 连续字母（所有 Unicode 字母，包括中日韩）
        \p{N}+|                   # 连续数字
        [^\s\p{L}\p{N}]+|         # 其他非空白、非字母、非数字（标点、符号等）
        \s+                       # 空白（包括空格、制表符、换行）
    """,
    re.X,
)



In [28]:
## 全局变量
GLOBAL_DICT = []

In [36]:
def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))


In [29]:
def pre_tokenization(text: str) -> List[str]:
    """
    标准 pre_tokenization:
    1. 先用 SPECIAL_TOKEN_PATTERN 提取 <|endoftext|>，确保它不被拆开；
    2. 对其它普通片段用 GENERAL_TOKEN_PATTERN 切分；
    3. 返回 List[str]，每个元素是一个 pre-token（包括空白）。
    """
    tokens: List[str] = []
    pos = 0

    # 遍历所有 special token 的匹配
    for m in SPECIAL_TOKEN_PATTERN.finditer(text):
        start, end = m.span()

        # 1) special token 之前的普通文本
        if start > pos:
            chunk = text[pos:start]
            for tok in GENERAL_TOKEN_PATTERN.findall(chunk):
                if tok:  # 去掉空字符串
                    tokens.append(tok)

        # 2) special token 本身作为一个整体 token
        tokens.append(m.group(0))
        pos = end

    # 3) 最后一个 special token 之后剩余的文本
    if pos < len(text):
        chunk = text[pos:]
        for tok in GENERAL_TOKEN_PATTERN.findall(chunk):
            if tok:
                tokens.append(tok)

    return tokens

if __name__ == "__main__":
    # 准备一个包含中英文、数字、标点和 <|endoftext|> 的测试字符串
    test_str = "Hello, world! 你好，世界 12345 <|endoftext|> Next line.\n第二行 text's here."

    toks = pre_tokenization(test_str)

    print("原始字符串:")
    print(test_str)
    print("\npre_tokenization 结果:")
    for i, t in enumerate(toks):
        print(f"{i:2d}: {repr(t)}")

原始字符串:
Hello, world! 你好，世界 12345 <|endoftext|> Next line.
第二行 text's here.

pre_tokenization 结果:
 0: 'Hello'
 1: ','
 2: ' '
 3: 'world'
 4: '!'
 5: ' '
 6: '你好'
 7: '，'
 8: '世界'
 9: ' '
10: '12345'
11: ' '
12: '<|endoftext|>'
13: ' '
14: 'Next'
15: ' '
16: 'line'
17: '.'
18: '\n'
19: '第二行'
20: ' '
21: 'text'
22: "'s"
23: ' '
24: 'here'
25: '.'


In [30]:
from collections import defaultdict
from typing import Dict, Tuple, List

# corpus 结构:
# corpus[chunk_idx][token_idx][byte_pos] = int_id


def count_pair_frequencies(
    corpus: List[List[List[int]]]
) -> Dict[Tuple[int, int], int]:
    """
    在“pre-token 内部”统计相邻 token pair 的频率。
    不跨 pre-token 统计 pair。
    """
    pair_freq: Dict[Tuple[int, int], int] = defaultdict(int)

    for chunk in corpus:
        for tok_bytes in chunk:  # tok_bytes: List[int]
            if len(tok_bytes) < 2:
                continue
            for a, b in zip(tok_bytes[:-1], tok_bytes[1:]):
                pair_freq[(a, b)] += 1

    return pair_freq


def get_most_frequent_pair(
    pair_freq: Dict[Tuple[int, int], int]
) -> Tuple[Tuple[int, int] | None, int]:
    if not pair_freq:
        return None, 0
    best_pair, best_freq = max(pair_freq.items(), key=lambda x: x[1])
    return best_pair, best_freq


def merge_pair_in_corpus(
    corpus: List[List[List[int]]],
    pair: Tuple[int, int],
    new_token_id: int,
) -> List[List[List[int]]]:
    """
    在 corpus 中，把所有出现的给定 pair (a, b) 替换为 new_token_id。
    只在“pre-token 内部”进行合并。
    """
    a, b = pair
    new_corpus: List[List[List[int]]] = []

    for chunk in corpus:
        new_chunk: List[List[int]] = []
        for tok_bytes in chunk:
            if len(tok_bytes) < 2:
                new_chunk.append(tok_bytes)
                continue

            merged: List[int] = []
            i = 0
            while i < len(tok_bytes):
                if i < len(tok_bytes) - 1 and tok_bytes[i] == a and tok_bytes[i + 1] == b:
                    merged.append(new_token_id)
                    i += 2
                else:
                    merged.append(tok_bytes[i])
                    i += 1
            new_chunk.append(merged)
        new_corpus.append(new_chunk)

    return new_corpus

In [31]:
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple
import multiprocessing as mp

# 假定你已有:
# - NUM_PROCESSES
# - SPECIAL_TOKENS (注意：这是你原来传进 find_chunk_boundaries 的，保留用法不动)
# - find_chunk_boundaries
# 再加上上面实现的 pre_tokenization / count_pair_frequencies / merge_pair_in_corpus


@dataclass
class BPEResult:
    vocab: Dict[int, bytes]              # id -> bytes
    merges: List[Tuple[int, int]]        # merge 规则


def train_bpe(
    input_path: str,
    vocab_size: int,
    special_tokens: List[str],
) -> "BPEResult":
    # 1. 从 input_path 中读取文件（以 bytes 形式）
    with open(input_path, "rb") as f:
        # 2. 并发切分：find_chunk_boundaries 返回锚点下标     ex: [0,1024,2037,4096,...]
        boundaries = find_chunk_boundaries(f, NUM_PROCESSES, SPECIAL_TOKENS)

    # 2.a 准备 chunks (utf-8) 供并发处理
    chunks: List[str] = []
    with open(input_path, "rb") as f2:
        for i, (start, end) in enumerate(zip(boundaries[:-1], boundaries[1:])):
            print(f"Loading chunk {i + 1} / {len(boundaries) - 1}")
            length_to_read = end - start
            f2.seek(start)
            raw_data = f2.read(length_to_read)  # bytes
            text_chunk = raw_data.decode("utf-8", errors="ignore")
            chunks.append(text_chunk)

    # 3. 多进程预处理：使用标准 pre_tokenization
    print("MultiProcess pre_tokenization Starting")
    with mp.Pool(processes=NUM_PROCESSES) as pool:
        # result: List[List[str]]，每个元素是一整个 chunk 的 pre-tokens
        tokenized_chunks: List[List[str]] = pool.map(pre_tokenization, chunks)

    # 3.a 归约（用于分析，BPE 主循环现在“间接”依赖这些边界）
    GLOBAL_DICT = Counter()
    for toks in tokenized_chunks:
        GLOBAL_DICT.update(toks)

    print(f"Total distinct pre-tokens: {len(GLOBAL_DICT)}")
    # 你可以按需打印高频 token：
    # print(GLOBAL_DICT.most_common(50))

    # 4. 构造“感知 pre-token 边界”的 byte-level 语料
    # corpus: List[List[List[int]]]
    #   = [chunk][token_idx][byte_id]
    corpus: List[List[List[int]]] = []
    for toks in tokenized_chunks:
        chunk_seq: List[List[int]] = []
        for tok in toks:
            byte_seq = tok.encode("utf-8")   # bytes
            token_ids = list(byte_seq)       # 每个 byte 是一个初始 id
            chunk_seq.append(token_ids)
        corpus.append(chunk_seq)

    # 5. 初始化 vocab：标准 BPE 从 256 个 byte 开始
    vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)}
    
    # 你传进来的 special_tokens 目前只登记，不参与 BPE 过程；
    # 后续可以在这里给它们加额外 id，然后在 encode 阶段优先识别。

    merges: List[Tuple[int, int]] = []

    # 6. BPE 训练循环（基于“pre-token 内部”的 pair 统计）
    while len(vocab) < vocab_size:
        # 6.1 统计 pair 频率（感知 pre-token 边界）
        pair_freq = count_pair_frequencies(corpus)

        # 6.2 找到频率最高的 pair
        best_pair, best_freq = get_most_frequent_pair(pair_freq)

        # 终止条件：没有可 merge 的 pair 或频率太低
        if best_pair is None or best_freq < 1:
            print("No more pairs to merge, stopping early.")
            break

        a, b = best_pair

        # 6.3 为该 pair 分配新的 token id
        new_token_id = len(vocab)

        # 6.4 构造新 token 的 bytes 表示，并加入 vocab
        new_token_bytes = vocab[a] + vocab[b]
        vocab[new_token_id] = new_token_bytes

        # 6.5 记录 merge 规则（以后 encode 用）
        merges.append(best_pair)
        print(f"Merge #{len(merges)}: {best_pair} -> id {new_token_id}, freq={best_freq}")

        # 6.6 在语料中把所有该 pair 替换为新 token（只在 pre-token 内部）
        corpus = merge_pair_in_corpus(corpus, best_pair, new_token_id)

    # 7. 打包返回结果
    return BPEResult(
        vocab=vocab,
        merges=merges,
    )

In [ ]:
if __name__ == "__main__":
    print(1)
    result: BPEResult = train_bpe(
    input_path=FILE_PATH,
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    )
    print("\n=== Training done ===")
    print(f"Final vocab size: {len(result.vocab)}")
    print(f"Merges count: {len(result.merges)}")

    # 打印前 20 条 merge 看看合理不
    print("\nTop 20 merges:")
    for i, pair in enumerate(result.merges[:20], 1):
        a, b = pair
        print(
            f"{i:02d}. pair={pair}, bytes={result.vocab[a]!r} + {result.vocab[b]!r} -> {result.vocab[len(result.vocab)-len(result.merges)+i-1]!r}"
        )

1
Loading chunk 1 / 32
Loading chunk 2 / 32
Loading chunk 3 / 32
Loading chunk 4 / 32
Loading chunk 5 / 32
Loading chunk 6 / 32
Loading chunk 7 / 32
Loading chunk 8 / 32
Loading chunk 9 / 32
Loading chunk 10 / 32
Loading chunk 11 / 32
Loading chunk 12 / 32
Loading chunk 13 / 32
Loading chunk 14 / 32
Loading chunk 15 / 32
Loading chunk 16 / 32
Loading chunk 17 / 32
Loading chunk 18 / 32
Loading chunk 19 / 32
Loading chunk 20 / 32
Loading chunk 21 / 32
Loading chunk 22 / 32
Loading chunk 23 / 32
Loading chunk 24 / 32
Loading chunk 25 / 32
Loading chunk 26 / 32
Loading chunk 27 / 32
Loading chunk 28 / 32
Loading chunk 29 / 32
Loading chunk 30 / 32
Loading chunk 31 / 32
Loading chunk 32 / 32
MultiProcess pre_tokenization Starting
Total distinct pre-tokens: 46725
